In [1]:
import csv
import json

with open("appointments.csv", "r", newline="", encoding="utf-8") as file:
    reader = csv.DictReader(file)

    print("Headers:", reader.fieldnames)

    rows = list(reader)

print("Total rows:", len(rows))


Headers: ['appointment_id', 'patient_id', 'clinic_id', 'booked_on', 'appt_date', 'age', 'gender', 'distance_km', 'fee_inr', 'booking_channel', 'visit_type', 'reminder_sent', 'prior_no_shows', 'status']
Total rows: 1020


In [2]:
def parse_row(row):
    converted = row.copy()

    converted["age"] = int(row["age"])
    converted["fee_inr"] = int(row["fee_inr"])
    converted["prior_no_shows"] = int(row["prior_no_shows"])
    converted["distance_km"] = float(row["distance_km"])

    return converted

In [3]:
accepted = []
rejected = []

for row in rows:
    try:
        accepted.append(parse_row(row))
    except ValueError:
        rejected.append(row)

print("Accepted rows:", len(accepted))
print("Rejected rows:", len(rejected))

with open("rejected_rows.json", "w", encoding="utf-8") as file:
    json.dump(rejected, file, indent=2)

print("Rejected rows saved successfully!")

Accepted rows: 988
Rejected rows: 32
Rejected rows saved successfully!


In [4]:
import pandas as pd
import numpy as np

df = pd.read_csv("appointments.csv")

print("Shape:", df.shape)
print("\nDataset Information:")
df.info()

print("\nMissing Values:")
print(df.isnull().sum())

print("\nDuplicate Rows:", df.duplicated().sum())

Shape: (1020, 14)

Dataset Information:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1020 entries, 0 to 1019
Data columns (total 14 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   appointment_id   1020 non-null   object 
 1   patient_id       1020 non-null   object 
 2   clinic_id        1020 non-null   object 
 3   booked_on        1020 non-null   object 
 4   appt_date        1019 non-null   object 
 5   age              1008 non-null   float64
 6   gender           1020 non-null   object 
 7   distance_km      1000 non-null   float64
 8   fee_inr          1020 non-null   int64  
 9   booking_channel  1020 non-null   object 
 10  visit_type       1020 non-null   object 
 11  reminder_sent    1020 non-null   object 
 12  prior_no_shows   1020 non-null   int64  
 13  status           1016 non-null   object 
dtypes: float64(2), int64(2), object(10)
memory usage: 111.7+ KB

Missing Values:
appointment_id      0
patient_id   

In [5]:
df = df.drop_duplicates()

print("Rows after removing duplicates:", len(df))

Rows after removing duplicates: 1000


In [6]:
df["booking_channel"] = df["booking_channel"].str.strip().str.title()

print("Unique booking channels:")
print(df["booking_channel"].unique())

print("Rows after cleaning booking_channel:", len(df))

Unique booking channels:
['Phone' 'Walk-In' 'App']
Rows after cleaning booking_channel: 1000


In [7]:
df["appt_date"] = pd.to_datetime(df["appt_date"], errors="coerce")
df["booked_on"] = pd.to_datetime(df["booked_on"])

print("Rows after date conversion:", len(df))
print("Missing appointment dates:", df["appt_date"].isnull().sum())

Rows after date conversion: 1000
Missing appointment dates: 6


In [8]:
df = df.dropna(subset=["appt_date", "status"])

print("Rows after dropping missing dates and status:", len(df))

Rows after dropping missing dates and status: 990


In [9]:
# Step A: Treat invalid ages as missing
df.loc[(df["age"] < 1) | (df["age"] > 100), "age"] = np.nan

print("Missing ages after validity check:", df["age"].isnull().sum())

# Step B: Fill missing values with median
age_median = df["age"].median()
distance_median = df["distance_km"].median()

df["age"] = df["age"].fillna(age_median).round().astype(int)
df["distance_km"] = df["distance_km"].fillna(distance_median)

print("Age median:", age_median)
print("Distance median:", distance_median)

print("\nMissing values after filling:")
print(df[["age", "distance_km"]].isnull().sum())

Missing ages after validity check: 17
Age median: 48.0
Distance median: 8.25

Missing values after filling:
age            0
distance_km    0
dtype: int64


In [10]:
# Distance outliers
q1_dist = df["distance_km"].quantile(0.25)
q3_dist = df["distance_km"].quantile(0.75)

iqr_dist = q3_dist - q1_dist

lower_dist = q1_dist - 1.5 * iqr_dist
upper_dist = q3_dist + 1.5 * iqr_dist

distance_capped = ((df["distance_km"] < lower_dist) |
                   (df["distance_km"] > upper_dist)).sum()

df["distance_km"] = np.clip(
    df["distance_km"], lower_dist, upper_dist
)


# Fee outliers
q1_fee = df["fee_inr"].quantile(0.25)
q3_fee = df["fee_inr"].quantile(0.75)

iqr_fee = q3_fee - q1_fee

lower_fee = q1_fee - 1.5 * iqr_fee
upper_fee = q3_fee + 1.5 * iqr_fee

fee_capped = ((df["fee_inr"] < lower_fee) |
              (df["fee_inr"] > upper_fee)).sum()

df["fee_inr"] = np.clip(
    df["fee_inr"], lower_fee, upper_fee
)

print("Distance capped:", distance_capped)
print("Fee capped:", fee_capped)

print("\nDistance limits:", lower_dist, upper_dist)
print("Fee limits:", lower_fee, upper_fee)

Distance capped: 26
Fee capped: 3

Distance limits: -7.9375 25.5625
Fee limits: 68.75 1218.75


In [11]:
df["distance_km"] = df["distance_km"].round(1)

df["fee_inr"] = df["fee_inr"].round().astype(int)

print(df[["age", "distance_km", "fee_inr"]].head())

print("\nLargest fee:", df["fee_inr"].max())

   age  distance_km  fee_inr
0   60         18.0      850
1   39         12.5      350
2   63          4.1      850
4   52          7.5      750
5   64         19.0      700

Largest fee: 1219


In [12]:
# Handle invalid age values
df.loc[(df["age"] < 1) | (df["age"] > 100), "age"] = np.nan

# Fill missing values
df["age"] = df["age"].fillna(df["age"].median()).round().astype(int)
df["distance_km"] = df["distance_km"].fillna(df["distance_km"].median())

In [13]:
# Calculate lead days
df["lead_days"] = (df["appt_date"] - df["booked_on"]).dt.days

# Convert no-show status into numeric values
df["no_show"] = (df["status"] == "No-Show").astype(int)

print(df[["booked_on", "appt_date", "lead_days", "status", "no_show"]].head())

print("\nLead days range:", df["lead_days"].min(), "to", df["lead_days"].max())
print("Total no-shows:", df["no_show"].sum())
print("No-show rate:", round(df["no_show"].mean() * 100, 1), "%")

   booked_on  appt_date  lead_days    status  no_show
0 2025-01-24 2025-01-25          1   No-Show        1
1 2025-05-27 2025-06-06         10  Attended        0
2 2025-03-24 2025-03-30          6  Attended        0
4 2025-04-18 2025-04-22          4  Attended        0
5 2024-11-23 2025-01-02         40   No-Show        1

Lead days range: 1 to 45
Total no-shows: 218
No-show rate: 22.0 %


In [14]:
# Select numerical features
X = df[["age", "distance_km", "fee_inr", "lead_days"]].to_numpy()

# Mean of each column
column_means = np.mean(X, axis=0)

print("Column means:", column_means)

Column means: [ 48.17373737   9.50090909 635.71414141  11.81919192]


In [15]:
mask = df["lead_days"] > 14

print("Bookings above 14 lead days:", mask.sum())
print("Share of bookings:", round(mask.mean() * 100, 1), "%")

Bookings above 14 lead days: 296
Share of bookings: 29.9 %


In [16]:
x = df["distance_km"].to_numpy()

z_scores = (x - np.mean(x)) / np.std(x)

unusual_distances = np.abs(z_scores) > 2

print("Distance values with |z-score| > 2:", unusual_distances.sum())

Distance values with |z-score| > 2: 53


In [18]:
import pandas as pd
import requests

clinics = pd.read_csv("clinics.csv")

print(clinics.head())
print("\nTotal clinics:", len(clinics))
print("\nColumns:", clinics.columns.tolist())

  clinic_id             clinic_name    city    tier  latitude  longitude
0       C01   Kothrud Family Clinic    Pune   Metro   18.5074    73.8077
1       C02    Hadapsar Care Centre    Pune   Metro   18.5089    73.9260
2       C03  Sitabuldi Health Point  Nagpur  Tier-2   21.1458    79.0882
3       C04    Malviya Nagar Clinic  Jaipur  Tier-2   26.8553    75.8150
4       C05   Vaishali Nagar Clinic  Jaipur  Tier-2   26.9124    75.7873

Total clinics: 8

Columns: ['clinic_id', 'clinic_name', 'city', 'tier', 'latitude', 'longitude']


In [19]:
temperatures = []

for _, row in clinics.iterrows():
    params = {
        "latitude": row["latitude"],
        "longitude": row["longitude"],
        "hourly": "temperature_2m",
        "forecast_days": 1
    }

    temp = None
    reason = None

    for attempt in range(2):
        try:
            response = requests.get(
                "https://api.open-meteo.com/v1/forecast",
                params=params,
                timeout=10
            )

            if response.status_code != 200:
                reason = f"HTTP status {response.status_code}"
                continue

            data = response.json()

            hourly = data.get("hourly", {})
            temp_values = hourly.get("temperature_2m", [])

            if temp_values:
                temp = temp_values[0]
                reason = None
                break
            else:
                reason = "Hourly temperature data missing"

        except requests.exceptions.RequestException as e:
            reason = str(e)

    temperatures.append(temp)

    if temp is None:
        print(f"Failed: {row['clinic_name']} - {reason}")

clinics["temp_c"] = temperatures

print(clinics[["clinic_name", "temp_c"]])

                clinic_name  temp_c
0     Kothrud Family Clinic    17.9
1      Hadapsar Care Centre    18.5
2    Sitabuldi Health Point    19.9
3      Malviya Nagar Clinic    17.0
4     Vaishali Nagar Clinic    17.0
5  Edappally Medical Centre    25.1
6        Vijay Nagar Clinic    19.1
7    Gomti Nagar Care Point    15.1


In [20]:
clinics.to_csv("clinics_enriched.csv", index=False)

print("\nFile saved: clinics_enriched.csv")
print("Total rows:", len(clinics))
print("Temperature missing:", clinics["temp_c"].isnull().sum())


File saved: clinics_enriched.csv
Total rows: 8
Temperature missing: 0


In [22]:
name_col = "name" if "name" in clinics.columns else "clinic_name"
print(f"Failed: {row[name_col]} - {reason}")

Failed: Gomti Nagar Care Point - None


In [23]:
columns = [
    "appointment_id", "patient_id", "clinic_id", "booked_on",
    "appt_date", "age", "gender", "distance_km", "fee_inr",
    "booking_channel", "visit_type", "reminder_sent",
    "prior_no_shows", "status", "lead_days", "no_show"
]

df = df.sort_values("appointment_id").copy()

df = df[columns]

print("Shape:", df.shape)
print("Columns:", df.columns.tolist())

Shape: (990, 16)
Columns: ['appointment_id', 'patient_id', 'clinic_id', 'booked_on', 'appt_date', 'age', 'gender', 'distance_km', 'fee_inr', 'booking_channel', 'visit_type', 'reminder_sent', 'prior_no_shows', 'status', 'lead_days', 'no_show']


In [24]:
# Format dates
df["booked_on"] = df["booked_on"].dt.strftime("%Y-%m-%d")
df["appt_date"] = df["appt_date"].dt.strftime("%Y-%m-%d")

# Check appointment IDs
print("Unique appointment IDs:", df["appointment_id"].nunique())

# Check missing values
print("\nMissing values:")
print(df.isnull().sum())

# Export clean dataset
df.to_csv("appointments_clean.csv", index=False)

print("\nappointments_clean.csv saved!")

Unique appointment IDs: 990

Missing values:
appointment_id     0
patient_id         0
clinic_id          0
booked_on          0
appt_date          0
age                0
gender             0
distance_km        0
fee_inr            0
booking_channel    0
visit_type         0
reminder_sent      0
prior_no_shows     0
status             0
lead_days          0
no_show            0
dtype: int64

appointments_clean.csv saved!


In [25]:
def sql_value(value):
    if pd.isna(value):
        return "NULL"
    elif isinstance(value, (int, float, np.integer, np.floating)):
        return str(value)
    else:
        return f"'{value}'"


sql_statements = []

# Clinics INSERT statements first
for _, row in clinics.iterrows():
    values = [sql_value(value) for value in row]
    sql_statements.append(
        f"INSERT INTO clinics VALUES ({','.join(values)});"
    )

# Appointments INSERT statements next
for _, row in df.iterrows():
    values = [sql_value(value) for value in row]
    sql_statements.append(
        f"INSERT INTO appointments VALUES ({','.join(values)});"
    )

# Save SQL file
with open("load_data.sql", "w", encoding="utf-8") as file:
    file.write("\n".join(sql_statements))

print("SQL file generated successfully!")
print("Total INSERT statements:", len(sql_statements))

SQL file generated successfully!
Total INSERT statements: 998


In [26]:
print("Appointments shape:", df.shape)
print("Unique IDs:", df["appointment_id"].nunique())
print("Missing values:", df.isnull().sum().sum())
print("Clinic rows:", len(clinics))
print("SQL statements:", len(sql_statements))

print("\nFirst SQL statements:")
for statement in sql_statements[:3]:
    print(statement)

Appointments shape: (990, 16)
Unique IDs: 990
Missing values: 0
Clinic rows: 8
SQL statements: 998

First SQL statements:
INSERT INTO clinics VALUES ('C01','Kothrud Family Clinic','Pune','Metro',18.5074,73.8077,17.9);
INSERT INTO clinics VALUES ('C02','Hadapsar Care Centre','Pune','Metro',18.5089,73.926,18.5);
INSERT INTO clinics VALUES ('C03','Sitabuldi Health Point','Nagpur','Tier-2',21.1458,79.0882,19.9);
